# Yaw, pitch and roll computation from smooth csv

In [3]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# calculate yaw, pitch and roll from smooth data and save csv

In [4]:


# ==========================================================
# INPUT / OUTPUT
# ==========================================================
INPUT_DIR  = r"D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Chase_CSVData\xyz_Smooth"
OUTPUT_DIR = r"D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Chase_CSVData\Body_Orientation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================================
# HELPERS
# ==========================================================
def normalize(v):
    n = np.linalg.norm(v, axis=1, keepdims=True)
    n[n == 0] = np.nan
    return v / n

def unwrap_nan(a):
    a = a.astype(float).copy()
    mask = ~np.isnan(a)
    a[mask] = np.unwrap(np.radians(a[mask])) * 180 / np.pi
    return a

# ==========================================================
# FIND FILES
# ==========================================================
csv_files = glob.glob(os.path.join(INPUT_DIR, "*_SMOOTH.csv"))
if not csv_files:
    raise FileNotFoundError("No SMOOTH CSV files found.")

# ==========================================================
# PROCESS
# ==========================================================
for path in csv_files:
    fname = os.path.basename(path)
    print("Processing:", fname)

    df = pd.read_csv(path)
    t = df["frame"].values

    bead   = df[["pt1_X","pt1_Y","pt1_Z"]].values
    head   = df[["pt2_X","pt2_Y","pt2_Z"]].values
    lwb    = df[["pt3_X","pt3_Y","pt3_Z"]].values
    rwb    = df[["pt4_X","pt4_Y","pt4_Z"]].values
    center = df[["center_X","center_Y","center_Z"]].values

    # ======================================================
    # CENTER-BASED ORIENTATION
    # ======================================================
    y_axis_c = normalize(head - center)      # forward
    x_axis_c = normalize(lwb - rwb)           # left-right
    z_axis_c = normalize(np.cross(x_axis_c, y_axis_c))
    x_axis_c = normalize(np.cross(y_axis_c, z_axis_c))  # re-orthogonalize

    yaw_c = unwrap_nan(np.degrees(np.arctan2(y_axis_c[:,1], y_axis_c[:,0])))
    pitch_c = unwrap_nan(np.degrees(
        np.arctan2(
            y_axis_c[:,2],
            np.sqrt(y_axis_c[:,0]**2 + y_axis_c[:,1]**2)
        )
    ))
    roll_c = unwrap_nan(np.degrees(
        np.arctan2(-x_axis_c[:,2], z_axis_c[:,2])
    ))

    # ======================================================
    # HEAD-BASED ORIENTATION
    # ======================================================
    y_axis_h = normalize(bead - head)         # forward
    x_axis_h = normalize(lwb - rwb)           # left-right
    z_axis_h = normalize(np.cross(x_axis_h, y_axis_h))
    x_axis_h = normalize(np.cross(y_axis_h, z_axis_h))  # re-orthogonalize

    yaw_h = unwrap_nan(np.degrees(np.arctan2(y_axis_h[:,1], y_axis_h[:,0])))
    pitch_h = unwrap_nan(np.degrees(
        np.arctan2(
            y_axis_h[:,2],
            np.sqrt(y_axis_h[:,0]**2 + y_axis_h[:,1]**2)
        )
    ))
    roll_h = unwrap_nan(np.degrees(
        np.arctan2(-x_axis_h[:,2], z_axis_h[:,2])
    ))

    # ======================================================
    # SANITY CHECK (IMPORTANT)
    # ======================================================
    N = len(t)
    assert len(yaw_c) == N
    assert len(pitch_c) == N
    assert len(roll_c) == N
    assert len(yaw_h) == N
    assert len(pitch_h) == N
    assert len(roll_h) == N

    # ======================================================
    # SAVE CSV
    # ======================================================
    out_df = pd.DataFrame({
        "frame": t,
        "yaw_center": yaw_c,
        "pitch_center": pitch_c,
        "roll_center": roll_c,
        "yaw_head": yaw_h,
        "pitch_head": pitch_h,
        "roll_head": roll_h
    })

    out_name = fname.replace("_SMOOTH.csv", "_ORIENTATION.csv")
    out_df.to_csv(os.path.join(OUTPUT_DIR, out_name), index=False)

    print("Saved:", out_name)

print("\n Orientation CSVs generated.")


Processing: Final_DLTdv87_Data_DLTdv8_data_Trial2_180rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_Trial2_180rpmxyzpts_ORIENTATION.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial2_200rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_Trial2_200rpmxyzpts_ORIENTATION.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial3_180rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_Trial3_180rpmxyzpts_ORIENTATION.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial4_400rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_Trial4_400rpmxyzpts_ORIENTATION.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial5_180rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_Trial5_180rpmxyzpts_ORIENTATION.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial5_400rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_Trial5_400rpmxyzpts_ORIENTATION.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial7_400rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_

# Display and save plots from orientation csv

In [5]:

# ==========================================================
# PATHS
# ==========================================================
INPUT_DIR = r"D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Chase_CSVData\Body_Orientation"
BASE_TRIAL_DIR = r"D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data"

# ==========================================================
# FIND ORIENTATION FILES
# ==========================================================
csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_ORIENTATION.csv")))
if not csv_files:
    raise FileNotFoundError("No *_ORIENTATION.csv files found.")

# ==========================================================
# PLOTTING
# ==========================================================
for path in csv_files:
    base = os.path.splitext(os.path.basename(path))[0]
    print("Plotting:", base)

    # ------------------------------------------------------
    # Extract Trial + RPM (ROBUST)
    # ------------------------------------------------------
    m = re.search(r"(Trial\d+_\d+rpm)", base)
    if not m:
        raise ValueError(f"Could not extract Trial+RPM from filename: {base}")

    trial_name = m.group(1)

    # ------------------------------------------------------
    # Create trial-specific ORIENTATION_PLOTS folder
    # ------------------------------------------------------
    save_dir = os.path.join(BASE_TRIAL_DIR, trial_name, "ORIENTATION_PLOTS")
    os.makedirs(save_dir, exist_ok=True)

    # ------------------------------------------------------
    # Load data
    # ------------------------------------------------------
    df = pd.read_csv(path)
    t = df["frame"].values

    # ================= CENTER =================
    plt.figure(figsize=(14, 6))
    plt.plot(t, df["yaw_center"], label="Yaw (center)", linewidth=2)
    plt.plot(t, df["pitch_center"], label="Pitch (center)", linewidth=2)
    plt.plot(t, df["roll_center"], label="Roll (center)", linewidth=2)
    plt.xlabel("Frame")
    plt.ylabel("Angle (°)")
    plt.title("Yaw, Pitch, Roll — CENTER BASED")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    plt.savefig(os.path.join(save_dir, f"{base}_CENTER.png"), dpi=300)
    plt.close()

    # ================= HEAD =================
    plt.figure(figsize=(14, 6))
    plt.plot(t, df["yaw_head"], label="Yaw (head)", linewidth=2)
    plt.plot(t, df["pitch_head"], label="Pitch (head)", linewidth=2)
    plt.plot(t, df["roll_head"], label="Roll (head)", linewidth=2)
    plt.xlabel("Frame")
    plt.ylabel("Angle (°)")
    plt.title("Yaw, Pitch, Roll — HEAD BASED")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    plt.savefig(os.path.join(save_dir, f"{base}_HEAD.png"), dpi=300)
    plt.close()

    print("Saved to:", save_dir)
    
print("\n Orientation plots saved trial + RPM wise.")


Plotting: Final_DLTdv87_Data_DLTdv8_data_Trial2_180rpmxyzpts_ORIENTATION
Saved to: D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial2_180rpm\ORIENTATION_PLOTS
Plotting: Final_DLTdv87_Data_DLTdv8_data_Trial2_200rpmxyzpts_ORIENTATION
Saved to: D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial2_200rpm\ORIENTATION_PLOTS
Plotting: Final_DLTdv87_Data_DLTdv8_data_Trial3_180rpmxyzpts_ORIENTATION
Saved to: D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial3_180rpm\ORIENTATION_PLOTS
Plotting: Final_DLTdv87_Data_DLTdv8_data_Trial4_400rpmxyzpts_ORIENTATION
Saved to: D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial4_400rpm\ORIENTATION_PLOTS
Plotting: Final_DLTdv87_Data_DLTdv8_data_Trial5_180rpmxyzpts_ORIENTATION
Saved to: D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial5_180rpm\ORIENTATION_PLOTS
Plotting: Final_DLTdv87_Data_DLTdv8_data_Trial5_400rpmxyzpts_ORIENTATION
Saved to: D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial5_400rpm\ORIENTATION_PLOTS
Plotting: Final_